<a href="https://colab.research.google.com/github/shavas1010/Cricket-Analysis/blob/main/Batting_analysis_complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
import numpy as np

deliveries=pd.read_csv("deliveries.csv")
matches=pd.read_csv("matches.csv")

d1=deliveries.copy()
d2=matches.copy()

d2=d2.rename(columns={"id":"match_id"})
d1["isdot"]=d1["batsman_runs"].apply(lambda x : 1 if x==0 else 0)

d1["isone"]=d1["batsman_runs"].apply(lambda x : 1 if x==1 else 0)
d1["istwo"]=d1["batsman_runs"].apply(lambda x : 1 if x==2 else 0)
d1["isthree"]=d1["batsman_runs"].apply(lambda x : 1 if x==3 else 0)
d1["isfour"]=d1["batsman_runs"].apply(lambda x : 1 if x==4 else 0)
d1["issix"]=d1["batsman_runs"].apply(lambda x : 1 if x==6 else 0)
d1["overno"]=d1["over"].apply(lambda x: x-1)

def phase(x):
  if x>=0 and x<=5.9:
    return "powerplay"
  elif x>=6 and x<=15.9:
    return "middle"
  else:
    return "death"
def avg(x,y):
  if y>0:
    return x/y
  else:
    return x/1
def bpd(x,y):
  if y>0:
    return x/y
  else:
    return x/1
def dot(x,y):
  return (x/y)*100
def bpb(x,y):
  if y>0:
    return x/y
  else:
    return x
d1["phase"]=d1["overno"].apply(lambda x:phase(x))
d=pd.merge(d1,d2,on="match_id",how="left")


def analysis(df,phase,venue,opp):
  df=df[(df.phase==phase) & (df.venue==venue) & (df.bowling_team==opp)].reset_index()
  runs=pd.DataFrame(df.groupby("batsman")["batsman_runs"].sum()).rename(columns={"batsman_runs":"runs"})
  balls=pd.DataFrame(df.groupby("batsman")["ball"].count()).rename(columns={"ball":"balls"})
  innings=pd.DataFrame(df.groupby("batsman")["match_id"].apply(lambda x: len(list(np.unique(x))))).rename(columns={"match_id":"innings"})
  dismissals=pd.DataFrame(df.groupby("batsman")["player_dismissed"].count()).rename(columns={"player_dismissed":"dismissals"})
  fours=pd.DataFrame(df.groupby("batsman")["isfour"].sum()).rename(columns={"isfour":"4s"})
  sixes=pd.DataFrame(df.groupby("batsman")["issix"].sum()).rename(columns={"issix":"6s"})
  dots=pd.DataFrame(df.groupby("batsman")["isdot"].sum()).rename(columns={"isdot":"dots"})
  df=pd.merge(runs,balls,on="batsman").merge(innings,on="batsman").merge(dismissals,on="batsman").merge(fours,on="batsman").merge(sixes,on="batsman").merge(dots,on="batsman").reset_index()
  df["SR"]=round((df["runs"]/df["balls"])*100,2)
  df["rpi"]=round((df["runs"]/df["innings"]),2)
  df["avg"]=df.apply(lambda x: avg(x["runs"],x["balls"]),axis=1)
  df["bpd"]=df.apply(lambda x: bpd(x["balls"],x["dismissals"]),axis=1)
  df["boundary"]=df["4s"]+df["6s"]
  df["bpb"]=df.apply(lambda x: bpb(x["balls"],x["boundary"]),axis=1)
  df["dot%"]=df.apply(lambda x: dot(x["dots"],x["balls"]),axis=1)


  if (phase=="powerplay"):
    wt_sr,wt_rpi,wt_bpb,wt_dot=0.24,0.1,0.3,0.37
    df["calc_SR"]=df["SR"].apply(lambda x:x*x)
    df["calc_rpi"]=df["rpi"].apply(lambda x:x*x)
    df["calc_bpb"]=df["bpb"].apply(lambda x:x*x)
    df["calc_dot"]=df["dot%"].apply(lambda x:x*x)

    sq_sr,sq_rpi,sq_bpb,sq_dot=np.sqrt(df[["calc_SR","calc_rpi","calc_bpb","calc_dot"]].sum(axis=0))
    df["calc_SR"]=df["calc_SR"].apply(lambda x:x/sq_sr)
    df["calc_rpi"]=df["calc_rpi"].apply(lambda x:x/sq_rpi)
    df["calc_bpb"]=df["calc_bpb"].apply(lambda x:x/sq_bpb)
    df["calc_dot"]=df["calc_dot"].apply(lambda x:x/sq_dot)
    df["calc_SR"]=df["calc_SR"].apply(lambda x:x*wt_sr)
    df["calc_rpi"]=df["calc_rpi"].apply(lambda x:x*wt_rpi)
    df["calc_bpb"]=df["calc_bpb"].apply(lambda x:x*wt_bpb)
    df["calc_dot"]=df["calc_dot"].apply(lambda x:x*wt_dot)

    maxsr,minsr=max(np.array(df["calc_SR"])),min(np.array(df["calc_SR"]))
    maxrpi,minrpi=max(np.array(df["calc_rpi"])),min(np.array(df["calc_rpi"]))
    maxbpb,minbpb=max(np.array(df["calc_bpb"])),min(np.array(df["calc_bpb"]))
    maxdot,mindot=min(np.array(df["calc_dot"])),max(np.array(df["calc_dot"]))

    df["bestSR"]=df["calc_SR"].apply(lambda x:(x-maxsr)*(x-maxsr))
    df["bestrpi"]=df["calc_rpi"].apply(lambda x:(x-maxrpi)*(x-maxrpi))
    df["bestbpb"]=df["calc_bpb"].apply(lambda x:(x-maxbpb)*(x-maxbpb))
    df["bestdot"]=df["calc_dot"].apply(lambda x:(x-maxdot)*(x-maxdot))

    df["worstSR"]=df["calc_SR"].apply(lambda x:(x-minsr)*(x-minsr))
    df["worstrpi"]=df["calc_rpi"].apply(lambda x:(x-minrpi)*(x-minrpi))
    df["worstbpb"]=df["calc_bpb"].apply(lambda x:(x-minbpb)*(x-minbpb))
    df["worstdot"]=df["calc_dot"].apply(lambda x:(x-mindot)*(x-mindot))

    df["bestscore"]=df.apply(lambda x:x["bestSR"]+x["bestrpi"]+x["bestbpb"]+x["bestdot"],axis=1)
    df["worstscore"]=df.apply(lambda x:x["worstSR"]+x["worstrpi"]+x["worstbpb"]+x["worstdot"],axis=1)

    df["score"]=df.apply(lambda x: x["worstscore"]/(x["worstscore"]+x["bestscore"]),axis=1)
    df=df[(df.runs>=30) & (df.innings>2)]
    print(df[["batsman","innings","runs","SR","rpi","boundary","score"]].sort_values(by="score",ascending=False).head(10))

  elif (phase=="middle"):
    wt_sr,wt_rpi,wt_bpd,wt_dot=0.12,0.19,0.25,0.44

    #Normalizing the values By TOPISIES METHOD

    df["calc_SR"]=df["SR"].apply(lambda x:x*x)
    df["calc_rpi"]=df["rpi"].apply(lambda x:x*x)
    df["calc_bpd"]=df["bpd"].apply(lambda x:x*x)
    df["calc_dot"]=df["dot%"].apply(lambda x:x*x)

    sq_sr,sq_rpi,sq_bpd,sq_dot=np.sqrt(df[["calc_SR","calc_rpi","calc_bpd","calc_dot"]].sum(axis=0))

    df["calc_SR"]=df["calc_SR"].apply(lambda x:x/sq_sr)
    df["calc_rpi"]=df["calc_rpi"].apply(lambda x:x/sq_rpi)
    df["calc_bpd"]=df["calc_bpd"].apply(lambda x:x/sq_bpd)
    df["calc_dot"]=df["calc_dot"].apply(lambda x:x/sq_sr)

    # Now we have normalized the values by squaring and taking the square root
    # Now multipy the value with criteria weight

    df["calc_SR"]=df["calc_SR"].apply(lambda x:x*wt_sr)
    df["calc_rpi"]=df["calc_rpi"].apply(lambda x:x*wt_rpi)
    df["calc_bpd"]=df["calc_bpd"].apply(lambda x:x*wt_bpd)
    df["calc_dot"]=df["calc_dot"].apply(lambda x:x*wt_dot)

    maxsr,minsr=max(np.array(df["calc_SR"])),min(np.array(df["calc_SR"]))
    maxrpi,minrpi=max(np.array(df["calc_rpi"])),min(np.array(df["calc_rpi"]))
    maxbpd,minbpd=max(np.array(df["calc_bpd"])),min(np.array(df["calc_bpd"]))
    maxdot,mindot=min(np.array(df["calc_dot"])),max(np.array(df["calc_dot"]))

    df["best_SR"]=df["calc_SR"].apply(lambda x:(x-maxsr)*(x-maxsr))
    df["best_rpi"]=df["calc_rpi"].apply(lambda x:(x-maxrpi)*(x-maxrpi))
    df["best_bpd"]=df["calc_bpd"].apply(lambda x:(x-maxbpd)*(x-maxbpd))
    df["best_dot"]=df["calc_dot"].apply(lambda x:(x-maxdot)*(x-maxdot))

    df["worst_SR"]=df["calc_SR"].apply(lambda x:(x-minsr)*(x-minsr))
    df["worst_rpi"]=df["calc_rpi"].apply(lambda x:(x-minsr)*(x-minsr))
    df["worst_bpd"]=df["calc_bpd"].apply(lambda x:(x-minsr)*(x-minsr))
    df["worst_dot"]=df["calc_dot"].apply(lambda x:(x-minsr)*(x-minsr))

    df["best_score"]=df.apply(lambda x: x["best_SR"]+x["best_rpi"]+x["best_bpd"]+x["best_dot"],axis=1)
    df["worst_score"]=df.apply(lambda x: x["worst_SR"]+x["worst_rpi"]+x["worst_bpd"]+x["worst_dot"],axis=1)
    df["score"]=df.apply(lambda x: x["worst_score"]/(x["worst_score"]+x["best_score"]),axis=1)
    df=df[(df.runs>=30) & (df.innings>2)]
    print(df[["batsman","innings","runs","SR","rpi","boundary","score"]].sort_values(by="score",ascending=False).head(10))

  else:
    wt_sr,wt_rpi,wt_bpb,wt_dot=0.21,0.09,0.27,0.44
    df["calc_SR"]=df["SR"].apply(lambda x:x*x)
    df["calc_rpi"]=df["rpi"].apply(lambda x:x*x)
    df["calc_bpb"]=df["bpb"].apply(lambda x:x*x)
    df["calc_dot"]=df["dot%"].apply(lambda x:x*x)

    sq_sr,sq_rpi,sq_bpb,sq_dot=np.sqrt(df[["calc_SR","calc_rpi","calc_bpb","calc_dot"]].sum(axis=0))
    df["calc_SR"]=df["calc_SR"].apply(lambda x:x/sq_sr)
    df["calc_rpi"]=df["calc_rpi"].apply(lambda x:x/sq_rpi)
    df["calc_bpb"]=df["calc_bpb"].apply(lambda x:x/sq_bpb)
    df["calc_dot"]=df["calc_dot"].apply(lambda x:x/sq_dot)
    df["calc_SR"]=df["calc_SR"].apply(lambda x:x*wt_sr)
    df["calc_rpi"]=df["calc_rpi"].apply(lambda x:x*wt_rpi)
    df["calc_bpb"]=df["calc_bpb"].apply(lambda x:x*wt_bpb)
    df["calc_dot"]=df["calc_dot"].apply(lambda x:x*wt_dot)

    maxsr,minsr=max(np.array(df["calc_SR"])),min(np.array(df["calc_SR"]))
    maxrpi,minrpi=max(np.array(df["calc_rpi"])),min(np.array(df["calc_rpi"]))
    maxbpb,minbpb=max(np.array(df["calc_bpb"])),min(np.array(df["calc_bpb"]))
    maxdot,mindot=min(np.array(df["calc_dot"])),max(np.array(df["calc_dot"]))

    df["bestSR"]=df["calc_SR"].apply(lambda x:(x-maxsr)*(x-maxsr))
    df["bestrpi"]=df["calc_rpi"].apply(lambda x:(x-maxrpi)*(x-maxrpi))
    df["bestbpb"]=df["calc_bpb"].apply(lambda x:(x-maxbpb)*(x-maxbpb))
    df["bestdot"]=df["calc_dot"].apply(lambda x:(x-maxdot)*(x-maxdot))

    df["worstSR"]=df["calc_SR"].apply(lambda x:(x-minsr)*(x-minsr))
    df["worstrpi"]=df["calc_rpi"].apply(lambda x:(x-minrpi)*(x-minrpi))
    df["worstbpb"]=df["calc_bpb"].apply(lambda x:(x-minbpb)*(x-minbpb))
    df["worstdot"]=df["calc_dot"].apply(lambda x:(x-mindot)*(x-mindot))

    df["bestscore"]=df.apply(lambda x:x["bestSR"]+x["bestrpi"]+x["bestbpb"]+x["bestdot"],axis=1)
    df["worstscore"]=df.apply(lambda x:x["worstSR"]+x["worstrpi"]+x["worstbpb"]+x["worstdot"],axis=1)

    df["score"]=df.apply(lambda x: x["worstscore"]/(x["worstscore"]+x["bestscore"]),axis=1)
    df=df[(df.runs>=30) & (df.innings>2)]

    print(df[["batsman","innings","runs","SR","rpi","boundary","score"]].sort_values(by="score",ascending=False).head(10))

print(d["bowling_team"].unique())
print(d["venue"].unique())
analysis(d,input("phase:"),input("Stadium:"),input("Opposition:"))






['Royal Challengers Bangalore' 'Sunrisers Hyderabad'
 'Rising Pune Supergiant' 'Mumbai Indians' 'Kolkata Knight Riders'
 'Gujarat Lions' 'Kings XI Punjab' 'Delhi Daredevils'
 'Chennai Super Kings' 'Rajasthan Royals' 'Deccan Chargers'
 'Kochi Tuskers Kerala' 'Pune Warriors' 'Rising Pune Supergiants']
['Rajiv Gandhi International Stadium, Uppal'
 'Maharashtra Cricket Association Stadium'
 'Saurashtra Cricket Association Stadium' 'Holkar Cricket Stadium'
 'M Chinnaswamy Stadium' 'Wankhede Stadium' 'Eden Gardens'
 'Feroz Shah Kotla' 'Punjab Cricket Association IS Bindra Stadium, Mohali'
 'Green Park' 'Punjab Cricket Association Stadium, Mohali'
 'Sawai Mansingh Stadium' 'MA Chidambaram Stadium, Chepauk'
 'Dr DY Patil Sports Academy' 'Newlands' "St George's Park" 'Kingsmead'
 'SuperSport Park' 'Buffalo Park' 'New Wanderers Stadium'
 'De Beers Diamond Oval' 'OUTsurance Oval' 'Brabourne Stadium'
 'Sardar Patel Stadium, Motera' 'Barabati Stadium'
 'Vidarbha Cricket Association Stadium, Jamtha'

KeyboardInterrupt: Interrupted by user